# Compute Returns

This notebook computes daily stock returns and NASDAQ index returns from the raw market data.

These returns are used for:
- market model estimation
- abnormal return calculation
- cumulative abnormal returns (CAR)
- volatility estimation

The notebook does **not modify raw data**. It produces processed datasets for downstream analysis.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Define file paths

In [2]:
prices_path = Path("../data/raw/market_data/prices_raw.csv")
index_path = Path("../data/raw/market_data/nasdaq_index_raw.csv")

output_prices_path = Path("../data/processed/market_data_with_returns.csv")
output_index_path = Path("../data/processed/index_returns.csv")

output_prices_path.parent.mkdir(parents=True, exist_ok=True)

## 2. Load datasets

In [3]:
prices = pd.read_csv(prices_path)
index_df = pd.read_csv(index_path)

print("prices shape:", prices.shape)
print("index shape:", index_df.shape)

prices shape: (13170, 8)
index shape: (1317, 7)


## 3. Parse and sort dates

Sorting is critical before computing returns.

In [4]:
prices["Date"] = pd.to_datetime(prices["Date"])
index_df["Date"] = pd.to_datetime(index_df["Date"])

prices = prices.sort_values(["ticker", "Date"]).reset_index(drop=True)
index_df = index_df.sort_values("Date").reset_index(drop=True)

## 4. Compute stock returns

We compute simple returns:

R_t = (P_t - P_{t-1}) / P_{t-1}

We use **Adjusted Close** to account for splits and dividends.

In [5]:
prices["return"] = (
    prices.groupby("ticker")["Adj Close"]
    .pct_change()
)

## 5. Compute market (NASDAQ) returns

In [6]:
index_df["return"] = index_df["Adj Close"].pct_change()

## 6. Inspect results

In [7]:
display(prices.head(10))
display(index_df.head(10))

,Date,ticker,Open,High,Low,Close,Adj Close,Volume,return
0,2015-06-29,AAPL,31.365000,31.617500,31.120001,31.132500,27.805971,196645600,NaN
1,2015-06-30,AAPL,31.392500,31.530001,31.215000,31.357500,28.006931,177482800,0.007227
2,2015-07-01,AAPL,31.725000,31.735001,31.497499,31.650000,28.268185,120955200,0.009328
3,2015-07-02,AAPL,31.607500,31.672501,31.442499,31.610001,28.232452,108844000,-0.001264
4,2015-07-06,AAPL,31.235001,31.557501,31.212500,31.500000,28.134211,112241600,-0.003480
5,2015-07-07,AAPL,31.472500,31.537500,30.942499,31.422501,28.064987,187787200,-0.002460
6,2015-07-08,AAPL,31.120001,31.160000,30.635000,30.642500,27.368326,243046400,-0.024823
7,2015-07-09,AAPL,30.962500,31.014999,29.805000,30.017500,26.810120,314380000,-0.020396
8,2015-07-10,AAPL,30.485001,30.962500,30.302500,30.820000,27.526863,245418000,0.026734
9,2015-07-13,AAPL,31.257500,31.440001,31.080000,31.415001,28.058290,165762000,0.019306


,Date,Adj Close,Close,High,Low,Open,Volume,return
0,2015-06-29,4958.470215,4958.470215,5051.009766,4956.229980,5021.209961,2025580000,NaN
1,2015-06-30,4986.870117,4986.870117,5008.759766,4968.259766,5000.149902,2034430000,0.005728
2,2015-07-01,5013.120117,5013.120117,5038.549805,4994.459961,5029.049805,1814560000,0.005264
3,2015-07-02,5009.209961,5009.209961,5027.470215,4990.740234,5024.299805,1490810000,-0.000780
4,2015-07-06,4991.939941,4991.939941,5020.709961,4960.930176,4963.799805,1741500000,-0.003448
5,2015-07-07,4997.459961,4997.459961,5001.990234,4902.209961,4993.759766,2132080000,0.001106
6,2015-07-08,4909.759766,4909.759766,4965.450195,4901.509766,4953.979980,1931520000,-0.017549
7,2015-07-09,4922.399902,4922.399902,4982.189941,4920.399902,4976.149902,1861600000,0.002574
8,2015-07-10,4997.700195,4997.700195,5008.049805,4966.509766,4981.240234,1590230000,0.015297
9,2015-07-13,5071.509766,5071.509766,5074.810059,5036.680176,5037.270020,1694140000,0.014769


## 7. Check for missing returns

The first observation per ticker should be NaN (expected).

In [8]:
print("Missing stock returns:", prices["return"].isna().sum())
print("Missing index returns:", index_df["return"].isna().sum())

Missing stock returns: 10
Missing index returns: 1


## 8. Drop initial NaNs

These correspond to the first observation per ticker.

In [9]:
prices = prices.dropna(subset=["return"]).reset_index(drop=True)
index_df = index_df.dropna(subset=["return"]).reset_index(drop=True)

## 9. Basic sanity checks

In [10]:
print("Stock return summary:")
print(prices["return"].describe())

print("\nIndex return summary:")
print(index_df["return"].describe())

Stock return summary:
count    13160.000000
mean         0.001516
std          0.024580
min         -0.242291
25%         -0.008246
50%          0.001315
75%          0.011587
max          0.522901
Name: return, dtype: float64

Index return summary:
count    1316.000000
mean        0.000681
std         0.013387
min        -0.123213
25%        -0.003926
50%         0.001101
75%         0.006853
max         0.093460
Name: return, dtype: float64


## 10. Check extreme values

Very large returns may indicate data issues.

In [11]:
extreme_stock = prices[prices["return"].abs() > 0.5]
extreme_index = index_df[index_df["return"].abs() > 0.5]

print("Extreme stock returns:", len(extreme_stock))
print("Extreme index returns:", len(extreme_index))

Extreme stock returns: 1
Extreme index returns: 0


In [12]:
extreme_stock[["ticker", "Date", "Adj Close", "return"]]

,ticker,Date,Adj Close,return
1521,AMD,2016-04-22,3.99,0.522901


https://finance.yahoo.com/news/why-did-amd-stock-jump-154505561.html?guccounter=1&guce_referrer=aHR0cHM6Ly93d3cuZ29vZ2xlLmNvbS8&guce_referrer_sig=AQAAAM1F4GNn0mOGqgSmfZJJ4DhKVdTwF_aEBhpxSVqgwpzm9vJ8ogj0ik91O4ZJdjercZuwyD0x8d77fFRSyHClrnR4zHUCUSp5jnNXoaGiVh90FqDcLIxYw9ISjhxBwoBCSr-K0bOyZLpUeVqrautRM2kb-9n6gxjhw1cvKYE-7nCn corresponds to this event

## 11. Save processed datasets

In [13]:
prices.to_csv(output_prices_path, index=False)
index_df.to_csv(output_index_path, index=False)

print("Saved:")
print(output_prices_path)
print(output_index_path)

Saved:
../data/processed/market_data_with_returns.csv
../data/processed/index_returns.csv


## 12. Summary

This notebook successfully computed:

- Daily stock returns (by ticker)
- Daily NASDAQ index returns

These datasets are now ready for:
- event window construction
- market model estimation
- abnormal return calculation